In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
from sklearn.feature_selection import SelectKBest, f_classif
from imblearn.under_sampling import RandomUnderSampler

# Load dataset
df = pd.read_csv('/content/drive/MyDrive/dataset/malicious_2021.csv', low_memory=False)

# Select features and target columns
features = ['Querylength', 'domain_token_count', 'path_token_count', 'avgdomaintokenlen', 'longdomaintokenlen',
            'avgpathtokenlen', 'tld', 'charcompvowels', 'charcompace', 'ldl_url', 'ldl_domain', 'ldl_path',
            'ldl_filename', 'ldl_getArg', 'subDirLen', 'this.fileExtLen', 'ArgLen', 'pathurlRatio',
            'ArgUrlRatio', 'argDomanRatio', 'domainUrlRatio', 'pathDomainRatio', 'argPathRatio', 'executable',
            'isPortEighty', 'NumberofDotsinURL', 'ISIpAddressInDomainName', 'CharacterContinuityRate',
            'LongestVariableValue', 'URL_DigitCount', 'host_DigitCount', 'Directory_DigitCount',
            'File_name_DigitCount', 'Extension_DigitCount', 'Query_DigitCount', 'URL_Letter_Count',
            'host_letter_count', 'Directory_LetterCount', 'Filename_LetterCount', 'Extension_LetterCount',
            'Query_LetterCount', 'LongestPathTokenLength', 'Domain_LongestWordLength', 'Path_LongestWordLength',
            'sub-Directory_LongestWordLength', 'Arguments_LongestWordLength', 'URL_sensitiveWord',
            'URLQueries_variable', 'spcharUrl', 'delimeter_Domain', 'delimeter_path', 'delimeter_Count',
            'NumberRate_URL', 'NumberRate_Domain', 'NumberRate_DirectoryName', 'NumberRate_FileName',
            'NumberRate_Extension', 'NumberRate_AfterPath', 'SymbolCount_URL', 'SymbolCount_Domain',
            'SymbolCount_Directoryname', 'SymbolCount_FileName', 'SymbolCount_Extension', 'SymbolCount_Afterpath',
            'Entropy_URL', 'Entropy_Domain', 'Entropy_Filename', 'Entropy_Extension', 'Entropy_Afterpath']

# Clean dataset
df_cleaned = df.copy()
df_cleaned['tld'] = df_cleaned['tld'].astype(str)
df_cleaned['url_type'] = df_cleaned['url_type'].astype(str)
numeric_features = [f for f in features if f not in ['tld', 'url_type']]
df_cleaned = df_cleaned[np.isfinite(df_cleaned[numeric_features]).all(axis=1)]

# Encode 'tld' and 'url_type'
label_encoder_tld = LabelEncoder()
label_encoder_url_type = LabelEncoder()
df_cleaned['tld_encoded'] = label_encoder_tld.fit_transform(df_cleaned['tld'])
df_cleaned['url_type_encoded'] = label_encoder_url_type.fit_transform(df_cleaned['url_type'])

# Define features and labels for binary classification
X = df_cleaned[numeric_features + ['tld_encoded']]
y = df_cleaned['binary_label'] = df_cleaned['url_type'].apply(lambda x: 0 if x == 'benign' else 1)

# Feature selection
k_best_selector = SelectKBest(score_func=f_classif, k=25)
X_selected = k_best_selector.fit_transform(X, y)
selected_features = [X.columns[i] for i in k_best_selector.get_support(indices=True)]
X_selected_df = pd.DataFrame(X_selected, columns=selected_features)

# Apply undersampling
under_sampler = RandomUnderSampler(sampling_strategy={label: 40000 for label in y.unique()}, random_state=42)
X_resampled, y_resampled = under_sampler.fit_resample(X_selected_df, y)

# Train-test split for binary classification
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.3, random_state=42, stratify=y_resampled)

# Define hyperparameter grid
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

# Hyperparameter tuning with GridSearchCV for binary classification
rf_classifier = RandomForestClassifier(random_state=42)
grid_search_bin = GridSearchCV(estimator=rf_classifier, param_grid=param_grid, cv=3, n_jobs=-1, verbose=2)
grid_search_bin.fit(X_train, y_train)
best_rf_classifier_bin = grid_search_bin.best_estimator_

# Evaluate binary classification
y_test_pred_bin = best_rf_classifier_bin.predict(X_test)
print("Best Parameters for Binary Classification:", grid_search_bin.best_params_)
print("Binary Classification Report - Test Data (Tuned):")
print(classification_report(y_test, y_test_pred_bin))
print("Test Accuracy (Tuned):", accuracy_score(y_test, y_test_pred_bin))

# Multiclass classification for malicious URL types
malicious_df = df_cleaned[df_cleaned['binary_label'] == 1].copy()
X_multi = malicious_df[numeric_features + ['tld_encoded']]
y_multi = malicious_df['url_type_encoded']

# Feature selection for multiclass
k_best_selector_multi = SelectKBest(score_func=f_classif, k=25)
X_multi_selected = k_best_selector_multi.fit_transform(X_multi, y_multi)
selected_features_multi = [X_multi.columns[i] for i in k_best_selector_multi.get_support(indices=True)]
X_multi_selected_df = pd.DataFrame(X_multi_selected, columns=selected_features_multi)

# Train-test split for multiclass
X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    X_multi_selected_df, y_multi, test_size=0.4, random_state=42, stratify=y_multi
)

# Hyperparameter tuning with GridSearchCV for multiclass classification
grid_search_multi = GridSearchCV(estimator=rf_classifier, param_grid=param_grid, cv=3, n_jobs=-1, verbose=2)
grid_search_multi.fit(X_train_multi, y_train_multi)
best_rf_classifier_multi = grid_search_multi.best_estimator_

# Evaluate multiclass classification
y_test_pred_multi = best_rf_classifier_multi.predict(X_test_multi)
print("Best Parameters for Multiclass Classification:", grid_search_multi.best_params_)
print("Multiclass Classification Report - Test Data (Tuned):")
print(classification_report(y_test_multi, y_test_pred_multi))
print("Test Accuracy (Tuned):", accuracy_score(y_test_multi, y_test_pred_multi))

Fitting 3 folds for each of 216 candidates, totalling 648 fits
